In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# Detectamos si ejecutamos desde la raíz del proyecto o desde notebooks
if (PROJECT_ROOT / "ml" / "data" / "raw").exists():
    DATA_DIR = PROJECT_ROOT / "ml" / "data" / "raw"
elif (PROJECT_ROOT.parent / "data" / "raw").exists():
    DATA_DIR = PROJECT_ROOT.parent / "data" / "raw"
elif Path("/content").exists():
    DATA_DIR = Path("/content")
else:
    raise FileNotFoundError("Dataset directory not found.")

SPAPHISH_PATH = DATA_DIR / "spaphish" / "Spaphish dataset - DiB.csv"
SPAMSPA_TRAIN_PATH = DATA_DIR / "spamspa" / "train.csv"
SPAMSPA_TEST_PATH = DATA_DIR / "spamspa" / "test.csv"

for path in [SPAPHISH_PATH, SPAMSPA_TRAIN_PATH, SPAMSPA_TEST_PATH]:
    print(path.name, "->", path.exists())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

spamspa_train = pd.read_csv(SPAMSPA_TRAIN_PATH)
spamspa_test = pd.read_csv(SPAMSPA_TEST_PATH)

print("TRAIN dimensions:", spamspa_train.shape)
print("TRAIN columns:", spamspa_train.columns.tolist())

print("\nTEST dimensions:", spamspa_test.shape)
print("TEST columns:", spamspa_test.columns.tolist())


In [ ]:
def basic_audit(df, name):
    print(f"\n{'=' * 50}")
    print(f"DATASET: {name}")
    print(f"{'=' * 50}")

    print("\nDimensions:", df.shape)

    print("\nData types:")
    display(df.dtypes.to_frame("dtype"))

    print("\nMissing values:")
    missing = pd.DataFrame({
        "missing": df.isna().sum(),
        "percentage": (df.isna().mean() * 100).round(2)
    })
    display(missing)

    print("\nExact duplicate rows:", df.duplicated().sum())

    print("\nNumeric descriptive statistics:")
    display(df.select_dtypes(include="number").describe().T)

    print("\nLabel distribution:")
    if "label" in df.columns:
        display(df["label"].value_counts(dropna=False).to_frame("count"))
        display(
            (df["label"].value_counts(normalize=True, dropna=False) * 100)
            .round(2)
            .to_frame("percentage")
        )


basic_audit(spamspa_train, "SpamSpa - Train")
basic_audit(spamspa_test, "SpamSpa - Test")


In [ ]:
# Cargamos SpaPhish utilizando el separador correcto
spaphish = pd.read_csv(
    SPAPHISH_PATH,
    sep=",",
    encoding="utf-8",
    on_bad_lines="error"
)

print("Dimensions:", spaphish.shape)

print("\nColumns:")
print(spaphish.columns.tolist())

print("\nDataset structure:")
print(spaphish.dtypes.to_string())

In [ ]:
# Calculamos el porcentaje de duplicados
for name, df in [
    ("Train", spamspa_train),
    ("Test", spamspa_test)
]:
    duplicates = df.duplicated().sum()
    percentage = duplicates / len(df) * 100

    print(
        f"{name}: {duplicates} duplicates "
        f"({percentage:.2f}%)"
    )

# Comprobamos si existen textos compartidos entre train y test
train_texts = set(
    spamspa_train["text"].dropna().str.strip().str.lower()
)

test_texts = set(
    spamspa_test["text"].dropna().str.strip().str.lower()
)

overlap = train_texts.intersection(test_texts)

print("\nTexts appearing in both sets:", len(overlap))
print(
    "Percentage of test texts overlapping with train:",
    round(len(overlap) / len(test_texts) * 100, 2),
    "%"
)

# Inspeccionamos los registros sin texto
display(spamspa_train[spamspa_train["text"].isna()])


In [ ]:
# Distribución de las etiquetas
print("Distribución de Label:")
display(spaphish["Label"].value_counts(dropna=False))

print("\nPorcentajes:")
display(
    (spaphish["Label"].value_counts(normalize=True, dropna=False) * 100)
    .round(2)
)

# Valores nulos
print("\nValores nulos por columna:")
nulls = spaphish.isna().sum()
display(nulls[nulls > 0].sort_values(ascending=False))

# Duplicados
print("\nDuplicados:")
print("Filas completas:", spaphish.duplicated().sum())
print("Hashes repetidos:", spaphish["hash"].duplicated().sum())

# Comprobar si hay cuerpos o asuntos vacíos
print("\nAsuntos vacíos:", spaphish["subject"].isna().sum())
print("Cuerpos vacíos:", spaphish["body"].isna().sum())


In [ ]:
# No se muestran ejemplos de mensajes originales para evitar exponer contenido sensible.
# La distribución de etiquetas se consulta en la celda de auditoría anterior.


In [ ]:
# Comprobar si los nulos de adjuntos y URLs son coherentes
print(
    "Registros con url_count = 0 y urls informadas:",
    ((spaphish["url_count"] == 0) & spaphish["urls"].notna()).sum()
)

print(
    "Registros con url_count > 0 y urls ausentes:",
    ((spaphish["url_count"] > 0) & spaphish["urls"].isna()).sum()
)

In [ ]:

# Crear una copia de los textos para el análisis
spaphish_analysis = spaphish.copy()

spaphish_analysis["subject"] = spaphish_analysis["subject"].fillna("")
spaphish_analysis["body"] = spaphish_analysis["body"].fillna("")

spaphish_analysis["full_text"] = (
    spaphish_analysis["subject"] + " " +
    spaphish_analysis["body"]
).str.strip()

spaphish_analysis["text_length"] = (
    spaphish_analysis["full_text"].str.len()
)

print("Estadísticas de longitud:")
display(
    spaphish_analysis.groupby("Label")["text_length"]
    .describe()
    .round(1)
)

# Comparar la longitud de los mensajes por clase
spaphish_analysis.boxplot(
    column="text_length",
    by="Label",
    figsize=(8, 5)
)

plt.title("Longitud de mensajes según su clasificación")
plt.suptitle("")
plt.xlabel("Etiqueta (0 = legítimo, 1 = phishing)")
plt.ylabel("Número de caracteres")
plt.show()


In [ ]:
# Crear el dataset normalizado de SpaPhish
spaphish_clean = spaphish[["subject", "body", "Label"]].copy()

# Completar los asuntos vacíos y construir el texto
spaphish_clean["subject"] = spaphish_clean["subject"].fillna("")
spaphish_clean["body"] = spaphish_clean["body"].fillna("")

spaphish_clean["text"] = (
    spaphish_clean["subject"].str.strip() + " " +
    spaphish_clean["body"].str.strip()
).str.strip()

# Normalizar el nombre de la etiqueta
spaphish_clean = spaphish_clean.rename(columns={"Label": "label"})

# Eliminar registros sin contenido
spaphish_clean = spaphish_clean[
    spaphish_clean["text"].str.len() > 0
].copy()

# Añadir la procedencia
spaphish_clean["source"] = "spaphish"

# Conservar únicamente las columnas necesarias
spaphish_clean = spaphish_clean[
    ["text", "label", "source"]
].reset_index(drop=True)

print("Dimensiones:", spaphish_clean.shape)
print("\nDistribución de etiquetas:")
display(spaphish_clean["label"].value_counts())


In [ ]:
print("Dimensions:", spaphish_clean.shape)

print("\nMissing values:")
display(spaphish_clean.isna().sum())

print("\nEmpty texts:", (spaphish_clean["text"].str.strip() == "").sum())

print("\nDuplicate texts:", spaphish_clean["text"].duplicated().sum())

print("\nLabel distribution:")
display(
    spaphish_clean["label"].value_counts().sort_index()
)

print("\nText length statistics by label:")
text_lengths = spaphish_clean.assign(text_length=spaphish_clean["text"].str.len())
display(text_lengths.groupby("label")["text_length"].describe().round(1))

## Conclusiones del análisis y comprensión de los datos

Durante esta primera fase se han cargado y explorado los datasets seleccionados para el proyecto Avizor: SpaPhish y SpamSpa, además de identificar sus características principales y comprobar su calidad.

### SpaPhish

El dataset SpaPhish contiene 1.395 correos electrónicos en español y 47 columnas. Incluye información sobre el asunto y el cuerpo del mensaje, fechas, URLs, archivos adjuntos y anotaciones relacionadas con técnicas de ingeniería social.

La variable `Label` presenta dos categorías:
- `0`: correo legítimo (664 registros, 47,6 %).
- `1`: phishing (731 registros, 52,4 %).

La distribución está bastante equilibrada, lo que resulta favorable para desarrollar un primer clasificador binario.

En cuanto a la calidad de los datos:
- No se han encontrado filas duplicadas ni hashes repetidos.
- Hay 3 asuntos vacíos, pero todos los registros tienen cuerpo.
- Los valores ausentes en `attachments_types` y `attachments_sizes` corresponden a mensajes sin archivos adjuntos.
- Los valores ausentes en `urls` observados corresponden a registros con cero URLs.
- Se han detectado 24 fechas ausentes y algunos valores ausentes en columnas de justificación de las anotaciones.

El análisis de longitud muestra diferencias entre las clases: los mensajes legítimos tienen una longitud media de aproximadamente 1.802 caracteres, mientras que los mensajes de phishing tienen una media de unos 758 caracteres. Esta diferencia es descriptiva y no debe interpretarse como una regla suficiente para detectar phishing.

La comprobación de coherencia entre url_count y urls no ha detectado inconsistencias. Todos los registros con url_count = 0 tienen la columna urls vacía, mientras que los registros con url_count > 0 contienen información sobre las URLs. Por tanto, los valores ausentes de esta columna se consideran ausencias estructurales y no errores de los datos.

### SpamSpa

Los archivos de SpamSpa se distribuyen en dos conjuntos:
- Entrenamiento: 4.457 registros y 2 columnas (`text`, `label`).
- Test: 1.115 registros y 2 columnas.

La distribución de etiquetas es similar en ambos conjuntos:
- `0`: 3.859 registros en train y 966 en test.
- `1`: 598 registros en train y 149 en test.

La proporción aproximada es del 86,6 % para la clase 0 y del 13,4 % para la clase 1, por lo que existe un desequilibrio entre clases.

La inspección de ejemplos indica que:
- `0` representa mensajes normales (*ham*).
- `1` representa mensajes de spam.

Se han detectado 275 filas duplicadas en train y 25 en test. Además, 118 textos aparecen en ambos conjuntos, lo que representa aproximadamente el 10,83 % de los textos distintos del conjunto de test. Este solapamiento debe tenerse en cuenta, ya que puede producir una evaluación demasiado optimista si se utiliza directamente esta separación.

También se han encontrado dos textos vacíos en el conjunto de entrenamiento.

### Decisiones para el desarrollo del modelo

Los dos datasets no utilizan la misma definición de las clases positivas:

- SpaPhish distingue entre correos legítimos y phishing.
- SpamSpa distingue entre mensajes normales y spam.

Por tanto, no se combinarán directamente como si ambas etiquetas representaran el mismo fenómeno. El spam no equivale necesariamente a phishing.

Para el primer modelo de detección de phishing se utilizará SpaPhish, por ser un conjunto específicamente orientado a esa tarea y presentar una distribución equilibrada.

En la preparación del primer clasificador se utilizarán el asunto y el cuerpo del correo como contenido textual. No se incluirán las columnas de anotaciones, justificaciones o indicadores derivados de la clasificación, para evitar introducir información que pueda provocar fuga de datos (*data leakage*).

SpamSpa se conservará como una posible fuente para desarrollar posteriormente un clasificador complementario de spam.

### Limitaciones identificadas

- SpaPhish tiene un tamaño relativamente reducido para entrenar un modelo generalizable.
- SpamSpa presenta desequilibrio de clases, duplicados y solapamiento entre train y test.
- Las etiquetas de ambos datasets representan problemas diferentes.
- Los resultados obtenidos con estos datos no garantizan por sí solos un buen rendimiento en mensajes reales recibidos por personas mayores.

Será necesario evaluar el modelo con métricas como precision, recall y F1-score, prestando especial atención a los falsos negativos y falsos positivos.

